# 演習8. 並列化すると順序が崩れる

## シーン

演習7で、ボトルネックの段を2人にしたら 16 FPS → 28 FPS になりました。
そして **Show に届くフレームの順番が入れ替わりました。**

```
0 1 3 2 4 6 5 7 9 10 8 12 13 11 15 ...
```

これは**バグではありません。** 並列化とは、こういうことです。
フレーム2の推論に 100ms、フレーム3の推論に 20ms かかるなら、
2人で同時に処理している以上、3のほうが先に終わります。

```
Infer係A   [ フレーム2 を推論 100ms      ]
Infer係B   [ 3を推論 20ms ][ 4を推論 ]
q2 に入る順 :  3 ,  4 ,  2 , ...              ← 追い越しが起きる
```

この演習では、

- **順番が崩れると、具体的に何が壊れるのか**
- **どう直すのか**
- **直すといくら払うことになるのか**

を順に見ます。

## 8-1. 【予測クイズ】順番が崩れると何が壊れるか

「順番が前後するだけなら、たいしたことはない」と思うかもしれません。
**壊れ方は、想像よりずっと悪いことがあります。**

次のプログラムでは、各フレームに **`id`（何番目のフレームか）** と
**`box`（そのフレームの検出結果）** を持たせます。
そして Show 側を、よくある書き方にしておきます。

```cpp
for (int expected = 0; expected < N; expected++) {
    Frame f = q2.pop();
    // ここで expected 番のフレームに f.box を描く
}
```

Show 側は「**来た順 ＝ フレーム番号**」だと思い込んでいます。
直列だったころは、それで正しかったからです。

**実行する前に予測してください。**

- 30フレームのうち、何枚が「思い込みと違う」ことになるでしょうか
- そのとき画面には何が表示されるでしょうか

In [ ]:
%%writefile ex08a.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <vector>
#include <map>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5で組み立てたキュー（ここでは中身は読まなくてよい） ----
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v); lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop(); lk.unlock(); can_push_.notify_one();
        return v;
    }
private:
    std::queue<T> q_;
    std::size_t capacity_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int N = 30;
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }
int infer_ms(int i) { return 20 + (i % 3) * 40; }   // 20 / 60 / 100、平均 60

// 1フレーム分のデータ。id は何番目のフレームか、box は推論結果のつもり
struct Frame { int id; int box; };

int main() {
    BoundedQueue<Frame> q1(4), q2(4);

    std::thread reader([&] {
        for (int i = 0; i < N; i++) { wait_ms(10); q1.push({i, -1}); }
    });

    std::vector<std::thread> inferers;                 // Infer は2人
    for (int k = 0; k < 2; k++)
        inferers.emplace_back([&, k] {
            for (int i = k; i < N; i += 2) {
                Frame f = q1.pop();
                wait_ms(infer_ms(f.id));
                f.box = f.id * 100;                    // 「そのフレームの検出結果」
                q2.push(f);
            }
        });

    std::vector<int> arrived;
    int mismatch = 0;
    std::thread shower([&] {
        // Show 側は「来た順 ＝ フレーム番号」と思い込んでいる
        for (int expected = 0; expected < N; expected++) {
            Frame f = q2.pop();
            arrived.push_back(f.id);
            if (f.id != expected) mismatch++;          // 思い込みが外れた回数
            wait_ms(30);
        }
    });

    reader.join();
    for (auto& t : inferers) t.join();
    shower.join();

    std::cout << "Show に届いた順番 :\n  ";
    for (int v : arrived) std::cout << v << " ";
    std::cout << "\n\n";

    int worst = 0;
    for (int i = 0; i < N; i++) { int d = arrived[i] - i; if (d < 0) d = -d; if (d > worst) worst = d; }
    std::cout << "順番どおりでなかったフレーム = " << mismatch << " / " << N << "\n";
    std::cout << "本来の位置からのずれ（最大） = " << worst << " 個分\n\n";

    std::cout << "Show 側が思い込みで処理すると、こうなる（最初の8フレーム）:\n";
    for (int i = 0; i < 8; i++)
        std::cout << "  " << i << " 番のフレームに、box=" << arrived[i] * 100
                  << "（" << arrived[i] << " 番の結果）を描いてしまう"
                  << (arrived[i] == i ? "" : "   <-- ずれている") << "\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex08a.cpp -o ex08a && ./ex08a

### 結果 ―― 30枚中 24枚がずれる

```
順番どおりでなかったフレーム = 24 / 30
本来の位置からのずれ（最大） = 2 個分
```

「ずれは最大2個分」なのに、**24枚が影響を受けています。**
1か所の入れ替わりが、そこから先の対応関係を全部ずらすからです。

そして被害の中身を見てください。

```
  2 番のフレームに、box=300（3 番の結果）を描いてしまう   <-- ずれている
  3 番のフレームに、box=200（2 番の結果）を描いてしまう   <-- ずれている
```

**別のフレームの検出結果が貼り付いています。**

これが「順番が崩れる」ことの本当の怖さです。表示がガタつくだけではありません。

- 検出枠が、1つ前（または後）のフレームの位置に描かれる
- 動いているものを追いかけていれば、枠だけが飛ぶ
- 集計していれば、集計そのものが間違う

しかも **プログラムは正常に動き続けます。** エラーも出ません。
「なんとなく枠がズレているような気がする」という形でしか気づけません。

> **並列化して速くなったとき、いちばん先に疑うべきは順序。**

### どのくらい崩れるかは、何で決まるか

- **担当者の人数**（2人なら最大1個分の追い越し、3人なら2個分…）
- **処理時間のばらつき**（全部同じ時間なら、ほとんど崩れません）

つまり **「速くするために増やした人数」が、そのまま崩れの大きさ**になります。

## 8-2. 直し方 ―― 出口で並べ直す

いちばん素直な方法は、**最後の段で並べ直す**ことです。

考え方は郵便の仕分けと同じです。

- 「**次に出すべき番号**」を1つ覚えておく（`next`）
- 届いたフレームが `next` なら、**そのまま出す**。そして `next` を1つ進める
- `next` でなければ、**手元に預かっておく**
- 出したあと、預かりの中に次の番号があれば、**続けて出す**

コードにすると、これだけです。

```cpp
std::map<int, Frame> pending;    // 預かり場所。番号順に並ぶ
int next = 0;                    // 次に出すべき番号

Frame f = q2.pop();
pending[f.id] = f;                                  // いったん預かる

while (!pending.empty() && pending.begin()->first == next) {
    出す(pending.begin()->second);
    pending.erase(pending.begin());
    next++;
}
```

`std::map` を使っているのは、**キーの小さい順に並んでくれる**からです。
`pending.begin()` が「預かっている中でいちばん若い番号」になります。

この仕組みを **リオーダバッファ（reorder buffer）** と呼びます。

**実行する前に予測してください。**

- 並べ直すと、FPS は落ちるでしょうか
- レイテンシはどうなるでしょうか
- 手元に預かるフレームは、最大で何枚になるでしょうか

In [ ]:
%%writefile ex08b.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <vector>
#include <map>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5で組み立てたキュー（ここでは中身は読まなくてよい） ----
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v); lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop(); lk.unlock(); can_push_.notify_one();
        return v;
    }
private:
    std::queue<T> q_;
    std::size_t capacity_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int N = 30;
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }
int infer_ms(int i) { return 20 + (i % 3) * 40; }   // 20 / 60 / 100、平均 60

struct Frame { int id; steady_clock::time_point born; };

void run(bool reorder) {
    BoundedQueue<Frame> q1(4), q2(4);
    std::vector<int> shown;                  // 実際に表示した順
    long long lat = 0;
    std::size_t worst_pending = 0;

    auto t0 = steady_clock::now();

    std::thread reader([&] {
        for (int i = 0; i < N; i++) { wait_ms(10); q1.push({i, steady_clock::now()}); }
    });

    std::vector<std::thread> inferers;
    for (int k = 0; k < 2; k++)
        inferers.emplace_back([&, k] {
            for (int i = k; i < N; i += 2) { Frame f = q1.pop(); wait_ms(infer_ms(f.id)); q2.push(f); }
        });

    std::thread shower([&] {
        std::map<int, Frame> pending;        // まだ順番が来ていないフレームの置き場
        int next = 0;                        // 次に出すべきフレーム番号

        for (int i = 0; i < N; i++) {
            Frame f = q2.pop();

            if (!reorder) {                  // 並べ直さない：来た順にそのまま出す
                shown.push_back(f.id);
                wait_ms(30);
                lat += duration_cast<milliseconds>(steady_clock::now() - f.born).count();
                continue;
            }

            pending[f.id] = f;                                  // いったん預かる
            if (pending.size() > worst_pending) worst_pending = pending.size();

            // 先頭が「次に出すべき番号」になっているあいだ、出し続ける
            while (!pending.empty() && pending.begin()->first == next) {
                Frame g = pending.begin()->second;
                pending.erase(pending.begin());
                shown.push_back(g.id);
                wait_ms(30);
                lat += duration_cast<milliseconds>(steady_clock::now() - g.born).count();
                next++;
            }
        }
    });

    reader.join();
    for (auto& t : inferers) t.join();
    shower.join();
    int ms = duration_cast<milliseconds>(steady_clock::now() - t0).count();

    bool ok = true;
    for (int i = 0; i < N; i++) if (shown[i] != i) ok = false;

    std::cout << (reorder ? "【並べ直しあり】" : "【並べ直しなし】") << "\n";
    std::cout << "  表示した順  : ";
    for (int v : shown) std::cout << v << " ";
    std::cout << "\n";
    std::cout << "  順番は正しいか : " << (ok ? "OK" : "NG") << "\n";
    std::cout << "  所要時間 " << ms << " ms   " << std::fixed << std::setprecision(1)
              << (1000.0 * N / ms) << " FPS   レイテンシ "
              << std::setprecision(0) << ((double)lat / N) << " ms"
              << "   手元に預かった最大数 " << worst_pending << "\n\n";
}

int main() {
    run(false);
    run(true);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex08b.cpp -o ex08b && ./ex08b

### 結果 ―― 順番は直り、速さは落ちない

```
【並べ直しなし】 NG   1063 ms  28.2 FPS  レイテンシ 222 ms  預かり 0
【並べ直しあり】 OK   1063 ms  28.2 FPS  レイテンシ 253 ms  預かり 3
```

- **順番 … 0 1 2 3 4 ... と完全に直りました**
- **FPS … 28.2 のまま。1ミリも落ちていません**
- **レイテンシ … 222ms → 253ms。少しだけ悪化**
- **預かり … 最大3枚**

**スループットが落ちないのが肝心なところです。** 並べ直しは「待たせる」処理ですが、
待たせているあいだも**上流は動き続けています**。だから全体の流れは止まりません。

払っているのは次の2つです。

- **レイテンシ** … 番号が若いフレームを待つあいだ、後続は出せません
- **メモリ** … 預かっているフレームのぶん

どちらも、**追い越しの幅**（＝並列にした人数と、処理時間のばらつき）で決まります。
2人なら数枚。10人にすれば、その分だけ増えます。

### 並べ直しの「弱点」

この方法には、はっきりした弱点が1つあります。

**`next` 番のフレームが永久に来なかったら、そこで止まります。**

推論に失敗して1枚捨てた、エラーで飛ばした、というだけで、
後続が全部せき止められます。しかも**エラーは出ません。静かに止まります。**

対処が必要です。発展課題2で扱います。

## まとめ

**① 並列化すると順序は崩れる。例外はない**

崩れないのは「処理時間が完全に同じ」ときだけで、そんなことは現実には起きません。

**② 崩れて困るかどうかは、下流が何を仮定しているかで決まる**

「来た順 ＝ 番号」と思い込んでいるコードがあれば、そこが壊れます。
逆に、フレームが自分の番号を持ち歩いていて、下流がそれを見ているなら、
**表示の順番以外は壊れません。**

> **データに番号を持たせておくと、崩れても直せる。**
> **番号を持たせていないと、崩れたことにすら気づけない。**

これは、並列化する**前に**やっておくべき準備です。

**③ 直すなら、出口で並べ直す**

スループットは落ちません。払うのはレイテンシとメモリです。

**④ そもそも並列化しない、という選択もある**

順序が絶対に崩れてはいけない段は、1人でやらせる。
演習7で見たとおり、**その段がボトルネックでないなら、増やす意味はそもそもありません。**

## 発展課題

1. 手元に預かる枚数の最大値は、何で決まるでしょうか。
   **Infer を4人、6人に増やしたら、預かる枚数はどうなる**と思いますか。

2. あるフレームが**捨てられて、二度と来ない**場合を考えます
   （推論に失敗した、エラーで飛ばした、など）。
   8-2 の並べ直しはどうなるでしょうか。また、**どう対処すればよい**でしょうか。

3. `std::map` の代わりに、**固定長の配列**（リングバッファ）で預かり場所を作れるでしょうか。
   作れるとしたら、何を保証できていることが条件でしょうか。

4. いまは Show スレッドの中で並べ直しています。
   これを**並べ直し専用のスレッド**として独立させたら、何が良くなり、何が悪くなるでしょうか。

5. 別の直し方として、「**担当者ごとに入口と出口のキューを分けて、
   Read が交互に配り、Show が交互に回収する**」という案があります（演習7の発展課題7の案3）。
   これなら並べ直しは要りません。
   **8-2 の方法と比べて、速さはどうなる**でしょうか。どんなときに不利になるでしょうか。

6. 本番の3段（Read → Infer → Show）のうち、
   **順序が保たれていなくても構わない段**はあるでしょうか。
   段ごとに「順序が崩れると何が困るか」を書き出してみてください。